指数移動WLあり
TKEOなし
12ch

In [1]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from scipy.ndimage import binary_closing
from scipy.signal import butter, lfilter, lfilter_zi


# =====================================================
# settings
# =====================================================
mus = ['TA', 'RF', 'VL', 'BF']

TARGET_ROWS = 30000
alpha = 0.01

folder_003 = r"C:\Users\masay\Documents\EMG_OpenSource\003\merged_data_10kh"


# =====================================================
# causal bandpass filter
# =====================================================
def emg_lfilter(
    emg,
    fs=1000,
    lowcut=20,
    highcut=450,
    order=2
):

    emg = np.asarray(emg)

    nyq = fs / 2

    low = lowcut / nyq
    high = highcut / nyq

    b, a = butter(
        order,
        [low, high],
        btype='band'
    )

    zi = lfilter_zi(
        b,
        a
    ) * emg[0]

    emg_filt, _ = lfilter(
        b,
        a,
        emg,
        zi=zi
    )

    return emg_filt


# =====================================================
# exponential RMS envelope
# =====================================================
def emg_envelope_exp_rms(
    emg,
    alpha=0.03
):

    env = np.zeros_like(emg)

    for i in range(len(emg)):

        if i == 0:

            env[i] = abs(
                emg[i]
            )

        else:

            env[i] = np.sqrt(

                (1 - alpha)
                * env[i - 1] ** 2

                +

                alpha
                * emg[i] ** 2

            )

    return env




# =====================================================
# exponential waveform length
# =====================================================
def emg_wl_exp(
    x,
    alpha=0.01
):

    x = np.asarray(x)

    y = np.zeros_like(x)

    for i in range(1, len(x)):

        diff = abs(
            x[i] - x[i - 1]
        )

        y[i] = (

            (1 - alpha)
            * y[i - 1]

            +

            alpha
            * diff

        )

    return y


# =====================================================
# EMG processing
# =====================================================
def process_emg(df):

    raw_processed = []

    rms_processed = []

    raw_dict = {}

    for n in mus:

        raw = emg_lfilter(
            df[n].values
        )

        raw_dict[n] = raw

        rect = np.abs(
            raw
        )

        rms = emg_envelope_exp_rms(
            rect,
            alpha=0.03
        )

        raw_processed.append(
            raw
        )

        rms_processed.append(
            rms
        )

    raw_processed = np.array(
        raw_processed
    ).T

    rms_processed = np.array(
        rms_processed
    ).T

    wl_features = []

    for n in mus:

        wl = emg_wl_exp(
            raw_dict[n],
            alpha=alpha
        )

        wl_features.append(
            wl
        )

    wl_features = np.stack(
        wl_features,
        axis=1
    )

    X = np.concatenate(

        [

            raw_processed,   # 4ch
            rms_processed,   # 4ch
            wl_features      # 4ch

        ],

        axis=1

    )

    return X


# =====================================================
# storage
# =====================================================
train_data = {}

val_data = {}

skipped = []


# =====================================================
# dataset 003 subject split
# =====================================================
subjects_003 = set()

for f in os.listdir(folder_003):

    if not f.endswith(".csv"):
        continue

    m = re.search(
        r"P(\d{4})",
        f
    )

    if m:

        subjects_003.add(
            int(m.group(1))
        )

subjects_003 = sorted(
    list(subjects_003)
)

train_subj_003, val_subj_003 = train_test_split(

    subjects_003,

    test_size=0.2,

    random_state=42,

    shuffle=True

)

train_subj_003 = set(
    train_subj_003
)

val_subj_003 = set(
    val_subj_003
)

print(
    f"003 train subjects = {len(train_subj_003)}"
)

print(
    f"003 val subjects = {len(val_subj_003)}"
)

print(
    "train:",
    sorted(train_subj_003)
)

print(
    "val:",
    sorted(val_subj_003)
)


# =====================================================
# dataset 003
# =====================================================
files_003 = os.listdir(
    folder_003
)

for f in files_003:

    if not f.endswith(".csv"):
        continue

    path = os.path.join(
        folder_003,
        f
    )

    df = pd.read_csv(
        path
    )

    if len(df) != TARGET_ROWS:

        skipped.append(

            (
                f,
                len(df)
            )

        )

        continue

    m = re.search(
        r"P(\d{4})",
        f
    )

    if m is None:
        continue

    subj = int(
        m.group(1)
    )

    # =================================
    # stance
    # =================================
    stance = df["Stance"].values

    stance = binary_closing(
        stance,
        structure=np.ones(5)
    ).astype(int)

    # =================================
    # EMG
    # =================================
    X = process_emg(
        df
    )

    y = stance

    # =================================
    # train / val
    # =================================
    if subj in train_subj_003:

        train_data.setdefault(
            subj,
            []
        ).append(
            (X, y)
        )

    elif subj in val_subj_003:

        val_data.setdefault(
            subj,
            []
        ).append(
            (X, y)
        )


# =====================================================
# pack
# =====================================================
def pack(data_dict):

    X_all = []

    y_all = []

    for _, data in data_dict.items():

        X = np.stack(
            [d[0] for d in data]
        )

        y = np.stack(
            [d[1] for d in data]
        )

        X_all.append(
            X
        )

        y_all.append(
            y
        )

    return (

        np.concatenate(
            X_all,
            axis=0
        ),

        np.concatenate(
            y_all,
            axis=0
        )

    )


# =====================================================
# final arrays
# =====================================================
X_train, y_train = pack(
    train_data
)

X_val, y_val = pack(
    val_data
)


# =====================================================
# info
# =====================================================
print("\n====================")
print("DATASET 003 ONLY")
print("====================")

print(
    "TRAIN:",
    X_train.shape
)

print(
    "VAL:",
    X_val.shape
)

print(
    "channels:",
    X_train.shape[-1]
)

print(
    "train subjects:",
    len(train_data)
)

print(
    "val subjects:",
    len(val_data)
)

if skipped:

    print(
        "\nskipped:",
        len(skipped)
    )

    for s in skipped[:10]:

        print(s)

003 train subjects = 87
003 val subjects = 22
train: [7, 9, 10, 12, 33, 34, 35, 36, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 62, 64, 65, 66, 67, 68, 70, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 96, 98, 100, 101, 102, 103, 104, 109, 110, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 135]
val: [2, 11, 37, 38, 39, 45, 57, 63, 69, 72, 83, 91, 95, 97, 99, 105, 106, 107, 108, 111, 123, 134]

DATASET 003 ONLY
TRAIN: (173, 30000, 12)
VAL: (44, 30000, 12)
channels: 12
train subjects: 87
val subjects: 22

skipped: 1
('P0124_01_merged.csv', 26664)


In [2]:
np.savez(
    "opensource_emg_dataset_3_10kh_003.npz",
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val
)